<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_5_%D0%94%D0%BE%D0%B2%D0%BE%D0%B4%D0%BA%D0%B0_%D0%B4%D0%BE_%D0%BF%D1%80%D0%BE%D0%B4%D0%B0%D0%BA%D1%88%D0%B5%D0%BD%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.5. Доводка до продакшена

## Введение: от прототипа к промышленной системе

Поздравляю! Мы прошли невероятный путь. В Лекции 6.1 мы создавали игрушечного RAG-агента на чистом Python, отправляя HTTP-запросы к Ollama и вычисляя косинусную близость вручную. В Лекции 6.2 мы масштабировали систему: добавили загрузку файлов, умный чанкинг, векторную базу Chroma, память и логирование. В Лекции 6.3 мы перешли на LangChain, сократив код в 3–5 раз и сделав его декларативным. В Лекции 6.4 мы построили настоящего агента на LangGraph: он умеет рассуждать, вызывать инструменты, запоминать контекст и даже спрашивать разрешения перед опасными действиями.

Но в реальной эксплуатации этого недостаточно. Производственная система требует:

- **Точного поиска** — чтобы находить релевантные документы даже по редким терминам и артикулам.
- **Прозрачности** — чтобы понимать, почему агент принял то или иное решение.
- **Скорости** — чтобы пользователь не ждал ответа по 30 секунд.
- **Удобного интерфейса** — чтобы с системой могли работать не только разработчики.
- **Безопасности** — чтобы агент не выполнял опасные действия без контроля.

В этой лекции мы доведём нашу систему до **продакшен-уровня**. Мы шаг за шагом:

1. **Улучшим поиск** — добавим гибридный поиск (BM25 + эмбеддинги) и реранкинг с кросс-энкодером.
2. **Настроим мониторинг** — подключим логирование и трассировку через LangSmith.
3. **Ускорим работу** — внедрим асинхронные вызовы и кеширование.
4. **Добавим интерфейс** — сделаем веб-интерфейс на Streamlit и Telegram-бота.
5. **Обеспечим безопасность** — настроим контроль действий и валидацию ввода.
6. **Заглянем в будущее** — обсудим планирование и самооценку агента.

К концу лекции вы получите **готовый продукт**, который можно развернуть в реальном проекте. Поехали!

---

## Тема 1. Гибридный поиск и реранкинг (скрипт `advanced_search.py`)

### 1.1. Почему одного семантического поиска недостаточно

В предыдущих лекциях мы использовали только **семантический поиск** — преобразование текста в эмбеддинги и поиск по косинусной близости. Этот подход отлично работает для естественных языковых запросов: он понимает синонимы, контекст и общий смысл.

**Но у него есть серьёзные слабости:**

| Сценарий | Семантический поиск | Почему плохо |
|----------|---------------------|--------------|
| **Артикулы товаров** | `"A100-2024"` | Не понимает точных кодов, может найти похожие по смыслу, но не точное совпадение |
| **Редкие термины** | `"энтропия Шеннона"` | Если термин редко встречается в обучающих данных, эмбеддинг может его «размазать» |
| **Имена собственные** | `"Иван Петров"` | Может найти других людей с похожей профессией, но не конкретного человека |
| **Цифры и даты** | `"2024-03-15"` | Не понимает точных дат, может найти документы с похожими событиями |

**Пример из реальной жизни:**

- **Документ:** «Артикул X-100: робот-пылесос с лазерной навигацией»
- **Запрос пользователя:** «Найди артикул X-100»
- **Семантический поиск:** может найти документы про «роботы» и «пылесосы», но не обязательно тот, где есть `X-100`
- **Что нужно:** точное совпадение по ключевым словам

---

### 1.2. Добавление BM25 — классический текстовый поиск

**BM25 (Best Matching 25)** — это алгоритм поиска по ключевым словам, который используется в поисковых системах уже несколько десятилетий. Он оценивает релевантность документа запросу на основе:

- Частоты терминов в документе (TF — Term Frequency).
- Обратной частоты термина в коллекции (IDF — Inverse Document Frequency).
- Нормализации длины документа.

**Установка:**

```bash
pip install rank-bm25
```

**Создание BM25-индекса (часть скрипта `advanced_search.py`):**

```python
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any

def tokenize(text: str) -> List[str]:
    """Токенизация текста для BM25."""
    # Для русского текста лучше использовать pymorphy2 или razdel
    # Для демонстрации используем простой сплит
    return text.lower().split()

# Создаём индекс на основе наших чанков
corpus = [chunk["text"] for chunk in chunked_documents]
tokenized_corpus = [tokenize(doc) for doc in corpus]
bm25_index = BM25Okapi(tokenized_corpus)
```

**Поиск через BM25 (часть скрипта `advanced_search.py`):**

```python
def bm25_search(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Поиск по BM25."""
    tokenized_query = tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)
    
    # Сортируем по убыванию релевантности
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "text": corpus[idx],
            "score": scores[idx],
            "metadata": chunked_documents[idx]["metadata"]
        })
    
    return results
```

**Сравнение результатов:**

| Запрос | Семантический поиск (Chroma) | BM25 |
|--------|------------------------------|------|
| «Артикул X-100» | Находит документы про «роботы» | Находит точное совпадение «X-100» |
| «Код 404 ошибка» | Находит про «ошибки» | Находит «404» |
| «Иван Петров» | Находит людей с похожей профессией | Находит точное имя |

**Вывод:** BM25 отлично дополняет семантический поиск, особенно когда нужно точное совпадение.

---

### 1.3. Объединение результатов: гибридный поиск

Теперь объединим два подхода: семантический поиск (Chroma) и BM25. Самый простой и эффективный метод — **Reciprocal Rank Fusion (RRF)**.

**Как работает RRF:**

1. Каждый поиск выдаёт свой список результатов.
2. Каждому результату присваивается ранг (1 = лучший).
3. Оценка = `1 / (rank + k)`, где `k` — константа (обычно 60).
4. Суммируем оценки для каждого документа.
5. Сортируем по итоговой оценке.

**Реализация гибридного поиска (часть скрипта `advanced_search.py`):**

```python
def reciprocal_rank_fusion(
    semantic_results: List[Dict],
    bm25_results: List[Dict],
    k: int = 60
) -> List[Dict]:
    """
    Объединяет результаты двух поисков через RRF.
    """
    scores = {}
    
    # Оцениваем семантические результаты
    for rank, result in enumerate(semantic_results, start=1):
        doc_id = result["metadata"]["chunk_index"]  # или другой уникальный ID
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rank + k)
    
    # Оцениваем BM25 результаты
    for rank, result in enumerate(bm25_results, start=1):
        doc_id = result["metadata"]["chunk_index"]
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rank + k)
    
    # Сортируем по убыванию
    sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    # Возвращаем топ-K кандидатов
    return sorted_results[:10]  # топ-10 кандидатов
```

**Полная функция гибридного поиска (часть скрипта `advanced_search.py`):**

```python
def hybrid_search(
    query: str,
    semantic_retriever,
    bm25_index,
    top_k: int = 3,
    alpha: float = 0.5
) -> List[Dict[str, Any]]:
    """
    Гибридный поиск с взвешенным суммированием.
    
    Аргументы:
        query: поисковый запрос
        semantic_retriever: ретривер Chroma
        bm25_index: BM25-индекс
        top_k: количество возвращаемых результатов
        alpha: вес семантического поиска (0-1)
    """
    # 1. Семантический поиск
    semantic_results = semantic_retriever.invoke(query, top_k=10)
    
    # 2. BM25 поиск
    tokenized_query = tokenize(query)
    bm25_scores = bm25_index.get_scores(tokenized_query)
    
    # 3. Нормализация оценок
    sem_scores = [1.0 - i/len(semantic_results) for i in range(len(semantic_results))]
    bm25_normalized = [s / max(bm25_scores) for s in bm25_scores]
    
    # 4. Объединение
    combined = {}
    for i, doc in enumerate(semantic_results):
        doc_id = doc.metadata.get("chunk_index")
        combined[doc_id] = {
            "text": doc.page_content,
            "metadata": doc.metadata,
            "score": alpha * sem_scores[i] + (1 - alpha) * bm25_normalized[i]
        }
    
    # 5. Сортировка
    sorted_results = sorted(combined.values(), key=lambda x: x["score"], reverse=True)
    return sorted_results[:top_k]
```

---

### 1.4. Реранкинг с кросс-энкодером

Гибридный поиск уже даёт хорошие результаты, но мы можем сделать ещё лучше. **Кросс-энкодер** — это модель, которая оценивает **пару** (вопрос, документ) и выдаёт оценку релевантности от 0 до 1. В отличие от эмбеддингов, кросс-энкодер учитывает взаимодействие между словами вопроса и документа, что даёт более точную оценку.

**Установка модели:**

```bash
pip install sentence-transformers
```

**Использование кросс-энкодера (часть скрипта `advanced_search.py`):**

```python
from sentence_transformers import CrossEncoder

# Загружаем лёгкую модель кросс-энкодера
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_with_cross_encoder(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    """
    Реранкинг кандидатов с помощью кросс-энкодера.
    """
    if not candidates:
        return []
    
    # Подготавливаем пары (вопрос, документ)
    pairs = [(query, doc["text"]) for doc in candidates]
    
    # Получаем оценки от кросс-энкодера
    scores = cross_encoder.predict(pairs)
    
    # Добавляем оценки в результаты
    for i, doc in enumerate(candidates):
        doc["rerank_score"] = float(scores[i])
    
    # Сортируем по оценке
    sorted_results = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    
    return sorted_results[:top_k]
```

**Полный пайплайн (часть скрипта `advanced_search.py`):**

```python
def advanced_search(query: str, top_k: int = 3) -> List[Dict]:
    """
    Полный пайплайн поиска с гибридным поиском и реранкингом.
    """
    # 1. Гибридный поиск — получаем топ-20 кандидатов
    candidates = hybrid_search(
        query=query,
        semantic_retriever=retriever,
        bm25_index=bm25_index,
        top_k=20
    )
    
    # 2. Реранкинг с кросс-энкодером — оставляем топ-3
    results = rerank_with_cross_encoder(query, candidates, top_k=top_k)
    
    return results
```

---

### 1.5. Тестовый пример

Давайте сравним подходы на реальном примере.

**Документы в базе:**
1. «Робот-пылесос X-100 с лазерной навигацией и функцией влажной уборки»
2. «Робот-пылесос X-200 с камерой и ИИ-распознаванием объектов»
3. «Артикул X-100: технические характеристики и инструкция по эксплуатации»
4. «Сравнение роботов-пылесосов: X-100 против X-200»

**Запрос пользователя:** «Артикул X-100»

**Результаты:**

| Подход | Топ-1 | Топ-2 | Топ-3 |
|--------|-------|-------|-------|
| **Семантический** | «Робот-пылесос X-100...» | «Сравнение X-100 и X-200» | «Робот-пылесос X-200...» |
| **BM25** | «Артикул X-100: характеристики» | «Робот-пылесос X-100...» | «Сравнение X-100 и X-200» |
| **Гибридный** | «Артикул X-100: характеристики» | «Робот-пылесос X-100...» | «Сравнение X-100 и X-200» |
| **+ Реранкинг** | «Артикул X-100: характеристики» (оценка 0.92) | «Робот-пылесос X-100...» (0.78) | «Сравнение X-100 и X-200» (0.65) |

**Вывод:**
- Чистый семантический поиск нашёл релевантные документы, но не самый точный.
- BM25 нашёл точное совпадение по ключевому слову.
- Гибридный поиск объединил сильные стороны обоих подходов.
- Реранкинг с кросс-энкодером дополнительно улучшил порядок результатов.

---

### 1.6. Полный код `advanced_search.py`

Сохраните следующий код в файл **`advanced_search.py`**:

```python
"""
advanced_search.py - Гибридный поиск с BM25 и реранкингом
Лекция 6.5, Тема 1
"""

import math
from typing import List, Dict, Any
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def tokenize(text: str) -> List[str]:
    """Токенизация текста для BM25."""
    return text.lower().split()

def bm25_search(query: str, bm25_index: BM25Okapi, corpus: List[str], top_k: int = 10) -> List[Dict]:
    """Поиск через BM25."""
    tokenized_query = tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"text": corpus[i], "score": scores[i]} for i in top_indices]

def hybrid_search(
    query: str,
    semantic_results: List[Dict],
    bm25_results: List[Dict],
    alpha: float = 0.5,
    top_k: int = 10
) -> List[Dict]:
    """
    Объединяет семантический поиск и BM25 с взвешенным суммированием.
    
    Аргументы:
        query: поисковый запрос
        semantic_results: результаты семантического поиска
        bm25_results: результаты BM25
        alpha: вес семантического поиска (0-1)
        top_k: количество возвращаемых результатов
    """
    # Создаём словари оценок
    sem_scores = {}
    for doc in semantic_results:
        doc_id = doc.get("metadata", {}).get("chunk_index", id(doc))
        sem_scores[doc_id] = doc.get("score", 1.0 / (semantic_results.index(doc) + 1))
    
    bm25_scores = {}
    for doc in bm25_results:
        doc_id = doc.get("metadata", {}).get("chunk_index", id(doc))
        bm25_scores[doc_id] = doc.get("score", 1.0 / (bm25_results.index(doc) + 1))
    
    # Нормализуем оценки в диапазон 0-1
    max_sem = max(sem_scores.values()) if sem_scores else 1
    max_bm25 = max(bm25_scores.values()) if bm25_scores else 1
    
    sem_scores = {k: v / max_sem for k, v in sem_scores.items()}
    bm25_scores = {k: v / max_bm25 for k, v in bm25_scores.items()}
    
    # Объединяем
    all_ids = set(sem_scores.keys()) | set(bm25_scores.keys())
    combined = {}
    
    # Создаём карту текстов
    text_map = {}
    for doc in semantic_results:
        doc_id = doc.get("metadata", {}).get("chunk_index", id(doc))
        text_map[doc_id] = doc.get("text", "")
    for doc in bm25_results:
        doc_id = doc.get("metadata", {}).get("chunk_index", id(doc))
        if doc_id not in text_map:
            text_map[doc_id] = doc.get("text", "")
    
    for doc_id in all_ids:
        sem_score = sem_scores.get(doc_id, 0)
        bm25_score = bm25_scores.get(doc_id, 0)
        
        combined[doc_id] = {
            "text": text_map.get(doc_id, ""),
            "metadata": {"chunk_index": doc_id},
            "score": alpha * sem_score + (1 - alpha) * bm25_score
        }
    
    sorted_results = sorted(combined.values(), key=lambda x: x["score"], reverse=True)
    return sorted_results[:top_k]

def rerank_with_cross_encoder(
    query: str,
    candidates: List[Dict],
    cross_encoder: CrossEncoder,
    top_k: int = 3
) -> List[Dict]:
    """
    Реранкинг кандидатов через кросс-энкодер.
    
    Аргументы:
        query: поисковый запрос
        candidates: список кандидатов
        cross_encoder: загруженный кросс-энкодер
        top_k: количество возвращаемых результатов
    """
    if not candidates:
        return []
    
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)
    
    for i, doc in enumerate(candidates):
        doc["rerank_score"] = float(scores[i])
    
    sorted_results = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return sorted_results[:top_k]

def advanced_search(
    query: str,
    semantic_retriever,
    bm25_index: BM25Okapi,
    corpus: List[str],
    cross_encoder: CrossEncoder,
    top_k: int = 3
) -> List[Dict]:
    """
    Полный пайплайн поиска с гибридным поиском и реранкингом.
    """
    # 1. Семантический поиск
    semantic_results = semantic_retriever.invoke(query, top_k=10)
    semantic_results = [
        {
            "text": doc.page_content,
            "metadata": doc.metadata,
            "score": 1.0 / (i + 1)
        }
        for i, doc in enumerate(semantic_results)
    ]
    
    # 2. BM25 поиск
    bm25_results = bm25_search(query, bm25_index, corpus, top_k=10)
    
    # 3. Гибридный поиск
    candidates = hybrid_search(query, semantic_results, bm25_results, alpha=0.5, top_k=10)
    
    # 4. Реранкинг
    final_results = rerank_with_cross_encoder(query, candidates, cross_encoder, top_k=top_k)
    
    return final_results


# ============================================================================
# Пример использования
# ============================================================================

if __name__ == "__main__":
    # Загрузка кросс-энкодера
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    
    # Тестовые данные
    corpus = [
        "Робот-пылесос X-100 с лазерной навигацией и функцией влажной уборки",
        "Робот-пылесос X-200 с камерой и ИИ-распознаванием объектов",
        "Артикул X-100: технические характеристики и инструкция по эксплуатации",
        "Сравнение роботов-пылесосов: X-100 против X-200"
    ]
    
    # Создаём BM25-индекс
    bm25_index = BM25Okapi([tokenize(doc) for doc in corpus])
    
    # Имитация ретривера (для демонстрации)
    class MockRetriever:
        def invoke(self, query, top_k=10):
            import random
            return [
                type('Doc', (), {'page_content': doc, 'metadata': {'chunk_index': i}})()
                for i, doc in enumerate(corpus)
                if random.random() > 0.3
            ][:top_k]
    
    retriever = MockRetriever()
    
    # Тестовый запрос
    query = "Артикул X-100"
    
    # Запуск поиска
    results = advanced_search(query, retriever, bm25_index, corpus, cross_encoder, top_k=3)
    
    print("Результаты после гибридного поиска и реранкинга:")
    for i, doc in enumerate(results, 1):
        print(f"{i}. {doc['text']} (оценка: {doc.get('rerank_score', 0):.2f})")
```

**Ожидаемый вывод:**

```
Результаты после гибридного поиска и реранкинга:
1. Артикул X-100: технические характеристики и инструкция по эксплуатации (оценка: 8.98)
2. Робот-пылесос X-100 с лазерной навигацией и функцией влажной уборки (оценка: 7.06)
3. Сравнение роботов-пылесосов: X-100 против X-200 (оценка: 5.12)
```

**Анализ результатов:**

| Место | Документ | Оценка кросс-энкодера |
|-------|----------|----------------------|
| 1 | Артикул X-100: технические характеристики | 8.98 |
| 2 | Робот-пылесос X-100 с лазерной навигацией | 7.06 |
| 3 | Сравнение роботов-пылесосов: X-100 против X-200 | 5.12 |

Кросс-энкодер правильно оценил, что документ с техническими характеристиками наиболее релевантен запросу «Артикул X-100». Документ с прямым упоминанием X-100 оказался на втором месте, а сравнение — на третьем.

---

## Краткий итог Тема 1

- Мы добавили **BM25** — классический поиск по ключевым словам, который отлично находит точные совпадения.
- Реализовали **гибридный поиск**, объединяющий семантический поиск и BM25 через взвешенное суммирование с нормализацией.
- Внедрили **реранкинг с кросс-энкодером** — модель, которая оценивает пару «вопрос + документ» и выдаёт точную оценку релевантности.
- Тестовый пример показал, что гибридный поиск + реранкинг находят самый точный документ, тогда как чистый семантический поиск может пропустить точное совпадение.

Теперь наш поиск стал значительно надёжнее. В следующей теме мы перейдём к **мониторингу и логированию** — чтобы понимать, как работает система в реальном времени и где её можно улучшить. Оставайтесь с нами!